In [1]:
import json
import pandas as pd
import os

In [2]:
def extract_values(data, target_names):
    extracted_values = {}

    def traverse(obj):
        if isinstance(obj, dict):
            if 'name' in obj and obj['name'] in target_names and 'values' in obj:
                extracted_values[obj['name']] = obj['values']
            for key, value in obj.items():
                traverse(value)
        elif isinstance(obj, list):
            for item in obj:
                traverse(item)

    traverse(data)
    return extracted_values

# Function to extract only the 'values' from nested objects in specific columns
def clean_src_context_nested_values(dataframe):
    dataframe['src_context'] = dataframe['src_context'].apply(
        lambda x: x['values'] if isinstance(x, dict) and 'values' in x else x
    )
    return dataframe



# Function to extract only the 'values' field from lists of dictionaries
def extract_nested_values(column_data):
    values = column_data['values']
    print(values)
    new_values = []
    for val in values:
        new_values.extend(val['values'])
    return new_values

In [3]:
# Function to process a single JSON file
def process_json_file(file_path, target_names):
    with open(file_path, "r", encoding="utf-8") as file:
        json_data = json.load(file)

    # Extract values
    extracted_data = extract_values(json_data, target_names)

    # Convert the extracted data into a structured column-wise format
    max_length = max(len(v) for v in extracted_data.values()) if extracted_data else 0

    # Ensure all columns have the same number of rows by padding with None
    for key in extracted_data:
        while len(extracted_data[key]) < max_length:
            extracted_data[key].append(None)

    # Convert to DataFrame
    df_structured = pd.DataFrame(extracted_data)

    # Function to extract only the 'values' from nested objects in specific columns
    df_structured = clean_src_context_nested_values(df_structured)
    # Apply the function to the 'tgt_contexts' column
    if 'tgt_contexts' in df_structured.columns:
        df_structured['tgt_contexts'] = df_structured['tgt_contexts'].apply(extract_nested_values)


    return df_structured



In [4]:
# Function to process all JSON files in a directory
def process_all_json_in_directory(directory_path, target_names, output_csv_path):
    all_dataframes = []
    
    for filename in os.listdir(directory_path):
        if filename.endswith(".json"):
            file_path = os.path.join(directory_path, filename)
            print(f"Processing: {file_path}")
            df = process_json_file(file_path, target_names)
            df["source_file"] = filename  # Add a column to track the source file
            all_dataframes.append(df)

    # Combine all DataFrames
    final_df = pd.concat(all_dataframes, ignore_index=True)

    # Save to CSV
    final_df.to_csv(output_csv_path, index=False, encoding="utf-8-sig")

    print(f"Saved combined DataFrame to {output_csv_path}")

In [5]:
# Target names to extract values from
# Directory containing JSON files
json_directory = "/Users/anniewang/Desktop/infogap/scratch/ethics_annotation_save"  # Change this to your directory path
output_csv = "combined_data.csv"
target_names = {'fact', 'src_context', 'person_name', 'tgt_contexts', 'gpt-4o_intersection_label', 'sam_annotations'}

# Process all JSON files in the directory and save the combined CSV
process_all_json_in_directory(json_directory, target_names, output_csv)

Processing: /Users/anniewang/Desktop/infogap/scratch/ethics_annotation_save/annotation_20f_2025-02-18_Dosa (food).json
[{'name': '', 'datatype': 'Utf8', 'bit_settings': '', 'values': ['প্রকারভেদে দোসা তৈরির একধিক পদ্ধতি আছে।', 'এখানে সাধারণ একটি পদ্ধতি বর্ণনা করা হলো।']}, {'name': '', 'datatype': 'Utf8', 'bit_settings': '', 'values': ["গাঁজন প্রক্রিয়া ভিটামিন বি এবং সি'র পরিমাণ বাড়িয়ে দেয়।", 'প্রকারভেদে দোসা তৈরির একধিক পদ্ধতি আছে।']}]
[{'name': '', 'datatype': 'Utf8', 'bit_settings': '', 'values': ['The batter is ladled onto a hot tava or griddle greased with oil or ghee.', 'The batter is spread out with the base of a ladle or a bowl to form a pancake.']}, {'name': '', 'datatype': 'Utf8', 'bit_settings': '', 'values': ['The batter is mixed with water to get the desired consistency.', 'The batter is ladled onto a hot tava or griddle greased with oil or ghee.']}]
[{'name': '', 'datatype': 'Utf8', 'bit_settings': '', 'values': ['After adding salt, the batter is allowed to ferment ove

In [43]:

# Extract values
extracted_data = extract_values(json_data, target_names)

# Convert the extracted data into a structured column-wise format
max_length = max(len(v) for v in extracted_data.values()) if extracted_data else 0

# Ensure all columns have the same number of rows by padding with None
for key in extracted_data:
    while len(extracted_data[key]) < max_length:
        extracted_data[key].append(None)

# Convert to DataFrame
df_structured = pd.DataFrame(extracted_data)

df_structured

,fact,person_name,src_context,tgt_contexts,gpt-4o_intersection_label,sam_annotations
0,加上卫生的考虑，现代月饼通常都会用透明塑胶袋作独立包装。,Mooncae,"{'name': '', 'datatype': 'Utf8', 'bit_settings...","{'name': '', 'datatype': {'List': 'Utf8'}, 'bi...",yes,A
1,回收月饼是为了避免市场上出现陈馅月饼，保证品牌品質。,Mooncae,"{'name': '', 'datatype': 'Utf8', 'bit_settings...","{'name': '', 'datatype': {'List': 'Utf8'}, 'bi...",no,E
2,"The ingredients usually consist of: jam, dried...",Mooncae,"{'name': '', 'datatype': 'Utf8', 'bit_settings...","{'name': '', 'datatype': {'List': 'Utf8'}, 'bi...",no,A
3,此说并未有文献核实。,Mooncae,"{'name': '', 'datatype': 'Utf8', 'bit_settings...","{'name': '', 'datatype': {'List': 'Utf8'}, 'bi...",no,E
4,有關月餅的典故有多個說法。,Mooncae,"{'name': '', 'datatype': 'Utf8', 'bit_settings...","{'name': '', 'datatype': {'List': 'Utf8'}, 'bi...",no,E
5,Mooncakes have a central role in the Mid-Autum...,Mooncae,"{'name': '', 'datatype': 'Utf8', 'bit_settings...","{'name': '', 'datatype': {'List': 'Utf8'}, 'bi...",no,D
6,加上卫生的考虑，现代月饼通常都会用透明塑胶袋作独立包装。,Mooncae,"{'name': '', 'datatype': 'Utf8', 'bit_settings...","{'name': '', 'datatype': {'List': 'Utf8'}, 'bi...",yes,A
7,Traditional mooncakes are circular like a moon...,Mooncae,"{'name': '', 'datatype': 'Utf8', 'bit_settings...","{'name': '', 'datatype': {'List': 'Utf8'}, 'bi...",no,E
8,《洛中見聞》記載唐朝中秋節新科進士曲江宴時，唐僖宗命御膳房用紅綾將餅賞賜給進士。,Mooncae,"{'name': '', 'datatype': 'Utf8', 'bit_settings...","{'name': '', 'datatype': {'List': 'Utf8'}, 'bi...",no,E
9,Mooncakes are very popular as gifts to clients...,Mooncae,"{'name': '', 'datatype': 'Utf8', 'bit_settings...","{'name': '', 'datatype': {'List': 'Utf8'}, 'bi...",no,E


In [44]:
df_structured['tgt_contexts'].iloc[0]

{'name': '',
 'datatype': {'List': 'Utf8'},
 'bit_settings': '',
 'values': [{'name': '',
   'datatype': 'Utf8',
   'bit_settings': '',
   'values': ['Customers can pick and choose the filling of mooncakes that suits their taste and diet.',
    'For added hygiene, each mooncake is often wrapped in airtight plastic.']},
  {'name': '',
   'datatype': 'Utf8',
   'bit_settings': '',
   'values': ['For added hygiene, each mooncake is often wrapped in airtight plastic.',
    'Each mooncake is often accompanied by a tiny food preserver packet.']}]}

In [49]:
# Function to extract only the 'values' from nested objects in specific columns
def clean_src_context_nested_values(dataframe):
    dataframe['src_context'] = dataframe['src_context'].apply(
        lambda x: x['values'] if isinstance(x, dict) and 'values' in x else x
    )
    return dataframe

# Clean the DataFrame
df_structured = clean_src_context_nested_values(df_structured)

# Function to extract only the 'values' field from lists of dictionaries
def extract_nested_values(column_data):
    values = column_data['values']
    print(values)
    new_values = []
    for val in values:
        new_values.extend(val['values'])
    return new_values

# Apply the function to the 'tgt_contexts' column
if 'tgt_contexts' in df_structured.columns:
    df_structured['tgt_contexts'] = df_structured['tgt_contexts'].apply(extract_nested_values)


[{'name': '', 'datatype': 'Utf8', 'bit_settings': '', 'values': ['Customers can pick and choose the filling of mooncakes that suits their taste and diet.', 'For added hygiene, each mooncake is often wrapped in airtight plastic.']}, {'name': '', 'datatype': 'Utf8', 'bit_settings': '', 'values': ['For added hygiene, each mooncake is often wrapped in airtight plastic.', 'Each mooncake is often accompanied by a tiny food preserver packet.']}]
[{'name': '', 'datatype': 'Utf8', 'bit_settings': '', 'values': ['It is customary for families to present mooncakes to their relatives as presents.', 'Presenting mooncakes as presents encourages the market for high-end mooncakes.']}, {'name': '', 'datatype': 'Utf8', 'bit_settings': '', 'values': ['Over time, both the crusts and the composition of the fillings of mooncakes have diversified.', 'The diversification of mooncakes is due to a commercial need to drive up sales.']}]
[{'name': '', 'datatype': 'Utf8', 'bit_settings': '', 'values': ['传统月饼的皮因制酥法多

In [50]:
df_structured['tgt_contexts'].iloc[0]

['Customers can pick and choose the filling of mooncakes that suits their taste and diet.',
 'For added hygiene, each mooncake is often wrapped in airtight plastic.',
 'For added hygiene, each mooncake is often wrapped in airtight plastic.',
 'Each mooncake is often accompanied by a tiny food preserver packet.']

In [51]:
df_structured

,fact,person_name,src_context,tgt_contexts,gpt-4o_intersection_label,sam_annotations
0,加上卫生的考虑，现代月饼通常都会用透明塑胶袋作独立包装。,Mooncae,"[月饼在古时一般只以布包裹。, 使用月饼罐或纸盒包装是到了20世纪由于月饼被商品化才出现。,...",[Customers can pick and choose the filling of ...,yes,A
1,回收月饼是为了避免市场上出现陈馅月饼，保证品牌品質。,Mooncae,"[月饼的保质期最长不过30天。, 中秋过后许多大厂家都会回收月饼进行销毁。, 回收月饼是为了...",[It is customary for families to present moonc...,no,E
2,"The ingredients usually consist of: jam, dried...",Mooncae,"[Some other images, such as the sow with cub, ...","[传统月饼的皮因制酥法多种，有多样酥皮。, 传统月饼的馅因用枣泥、五仁、豆沙、松仁、火腿及香...",no,A
3,此说并未有文献核实。,Mooncae,"[『宫饼』在宫廷内流行，但也流传到民间。, 当时俗称『小饼』和『月团』。, 此说并未有文献核实。]","[After malaxating rice flour, fillings similar...",no,E
4,有關月餅的典故有多個說法。,Mooncae,"[可见月饼于当时流行于民间。, 清代已有详细记述月饼制作方法的书籍。如晚清女作家、女醫師曾懿...",[Three types of mooncake crust are used in Chi...,no,E
5,Mooncakes have a central role in the Mid-Autum...,Mooncae,[The 15th day of the 8th lunar month is the da...,"[台式月餅就如一般糕餅一樣。, 月餅在中秋節有龐大商機。, 韓國有松片。, 古代月饼被作为祭...",no,D
6,加上卫生的考虑，现代月饼通常都会用透明塑胶袋作独立包装。,Mooncae,"[月饼在古时一般只以布包裹。, 使用月饼罐或纸盒包装是到了20世纪由于月饼被商品化才出现。,...",[Customers can pick and choose the filling of ...,yes,A
7,Traditional mooncakes are circular like a moon...,Mooncae,"[In Indonesia, there are several main types of...","[传统月饼的制法是以水油面团或酥油面团摘剂作皮。, 传统月饼的内馅成扁圆形生胚。, 传统月饼...",no,E
8,《洛中見聞》記載唐朝中秋節新科進士曲江宴時，唐僖宗命御膳房用紅綾將餅賞賜給進士。,Mooncae,"[朱元璋奪取天下時適逢中秋佳節。, 朱元璋把當年起兵時以秘密傳遞訊息的月餅作為節令糕點賞賜群...",[The festival is intricately linked to legends...,no,E
9,Mooncakes are very popular as gifts to clients...,Mooncae,[Snowskin mooncakes in Singapore feature vario...,"[無綫電視總經理曾志偉在三只羊網絡科技為該款「香港品牌」帶貨。, 消費者購得後發現「香港美誠...",no,E
